# LightLogger Data Processing Pipeline

## Overview

This notebook documents a full end-to-end preprocessing and analysis pipeline for the LightLogger / FLIC dataset.

The pipeline converts raw multimodal recordings (world camera + Neon eye tracking) into structured, analyzable outputs:

- Cleaned world camera videos
- Egocentric gaze mappings (Neon → World)
- Virtually foveated (retinal-coordinate) videos
- Spatial Power Density (SPD) statistics
- Grouped visualizations (per subject and across subjects)

This pipeline is hybrid:

- **Python** → orchestration, file management, batching
- **MATLAB** → vision processing, foveation, SPD computation

---

## High-Level Pipeline

1. Generate World Videos  
2. Run Egocentric Mapping  
3. Generate Virtually Foveated Videos  
4. Compute SPDs  
5. Group SPDs (Per Subject)  
6. Group SPDs (Across Subjects)  

**Each step depends on outputs from the previous stage.**

---

## Import libraries

In [1]:
import importlib
import preprocessing_pipeline

# Step 0 — Data Acquisition, Transfer, and Verification

## Goal

Ensure all raw recordings are:
- Downloaded or transferred correctly
- Complete (no missing chunks/files)
- Properly paired (Neon ↔ World)
- Ready for downstream processing

This step is critical to avoid silent data corruption or misalignment later in the pipeline.

---

## What This Step Does (Detailed)

### 1. Data Download / Transfer

Steps Performed in this Stage:

- Download recordings via API (e.g., Pupil Cloud endpoints)
- Transfer `.zip` recordings from external storage
- Extract archives into the expected directory structure

Typical structure enforced:

```
FLIC_XXXX/ 
    activity/
        GKA/      (world chunks)
        Neon/     (eye tracking data)
```

### 2. File Integrity Checks

After transfer, the pipeline verifies:

- Required directories exist (`GKA`, `Neon`)
- Files are non-empty
- Expected number of chunk files is present
- No partial or truncated downloads


### 3. Video Integrity Validation

For world camera data:

- Ensures GKA chunks can be successfully read
- Verifies frame continuity (no corrupted segments)
- Confirms metadata (FPS, resolution) is consistent

This prevents downstream failures in:
- video stitching
- SPD computation

---


### Transfer Light Logger Videos to NAS

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.transfer_light_logger_recordings(subjects_to_transfer=[],
                                                        activities_to_transfer=[],
                                                        overwrite_existing=False, 
                                                        verbose=True)

### Download Pupil Labs Data fromt the Cloud

#### 1. Download the Neon Timeseries + Scene Videos

In [ ]:
importlib.reload(preprocessing_pipeline)
api_key: str = "" # TODO: Fill this in, but NEVER commit this. If it is committed, immediately deactivate it on Pupil Cloud
preprocessing_pipeline.download_pupil_cloud_recordings(api_key, 
                                                       dst_dir="/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026", 
                                                       subjects_to_download=[2006], 
                                                       activities_to_download=["gazeCalibration", "work", "walkIndoor", "chat", "walkOutdoor", "sitBiopond", "walkBiopond"],
                                                       verbose=True,
                                                       overwrite_existing=False
                                                      )

### 2. Ensure downloaded neon files are well formatted and have all required content

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_neon_integrity(verbose=True)

---

# Step 1 — Generate World Camera Videos

## Goal

Convert raw chunk-based recordings into a single continuous video per activity.

## Input

Raw dataset structure:

```
FLIC_XXXX/
    activity/
        GKA/
            chunk files
```
## Output

Processed video:

```
FLIC_XXXX/
    activity/
        GKA/
            W.avi
```

## What This Step Does 

For every subject and activity:

1. Locates the `GKA` directory containing chunked frames
2. Uses `video_io.world_chunks_to_video(...)` to:
   - Stitch chunks into a continuous video stream
   - Handle missing frames via interpolation (if enabled)
   - Convert timestamps into seconds
3. Optionally applies:
   - Debayering (raw sensor → RGB)
   - Color weighting (camera calibration)
   - Digital gain correction
4. Writes a lossless `.avi` (FFV1 encoding)

---

### Generate the World Videos

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_world_videos(overwrite_existing=False, verbose=True) 

# Step 2 — Verifying Neon ↔ World Pairing

This is one of the most important checks.

For each activity:

- Locate Neon recording directory
- Locate corresponding world video (or chunks)
- Ensure BOTH exist before proceeding

Typical logic:

```
- Neon path:
    activity/Neon/<recording_folder>
- World path:
    activity/GKA/
```

Assertions ensure:

- Both modalities exist
- Neither is empty
- They correspond to the same recording session

**When verbose is enabled, output images are displayed to visually verify the recordings are from the same session**

---



### Ensure the world videos and neon videos are correctly paired

In [ ]:
importlib.reload(preprocessing_pipeline)

# Return value is a dictionary of the recordings whose integrity was verified
videos_observed: dict[str, dict[str, bool]] = preprocessing_pipeline.verify_world_neon_pairing(verbose=True)

## 4. Run Egocentric Video Mapper
Note: you have to switch to the "egocentric_video_mapper" kernel to run this step

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_egocentric_mapper_results(overwrite_existing=False, verbose=True)

## 4. Virtually Foveate

### 1. First find the start ends of activities

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_tag_task_start_ends(overwrite_existing=False, verbose=True)

### 2. Virtually Foveate the videos

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_virtually_foveated_videos(overwrite_existing=False, verbose=True, activities_to_skip=set(["lunch", "phone"]))

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.verify_virtually_foveated_video_integrity(verbose=True, activities_to_skip=["phone", "lunch"])

## 5. Generate SPDs

### 1. Generate the Indinvidual SPDs and plots for all desired subjects and activitys 

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds(color_mode="L-M",
                                     overwrite_existing=False, 
                                     verbose=True, 
                                     activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                     subjects_to_skip=[2001, 2002, 2003, 2004, 2005])

Locating project "lightLoggerAnalysis" within "/Users/zacharykelly/Documents/MATLAB/projects".
  Found at "/Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis".
Local copy of ToolboxToolbox is up to date.
Updating "ToolboxRegistry".
Already up to date.
Updating "lightLoggerAnalysis".
Already up to date.
Locating project "lightLogger" within "/Users/zacharykelly/Documents/MATLAB/projects".
  Found at "/Users/zacharykelly/Documents/MATLAB/projects/lightLogger".
Updating "lightLogger".
Already up to date.
Updating "transparentTrack".
Updating 72ecb8d..f21d01c
error: Your local changes to the following files would be overwritten by merge:
	stages/makeFitVideo.m
Please commit your changes or stash them before you merge.
Aborting
Updating "bads".
Already up to date.
Updating "fshift".
Updating "gkaModelEye".
Updating 0b5b688..15cabea
error: Your local changes to the following files would be overwritten by merge:
	model/ellipse/eyePoseEllipseFit.m
Please commit your changes or s

  Local hook template more recent than local hook. Consider updating.


Checking for "fshift" local hook.
Checking for "gkaModelEye" local hook.
  Running local hook "/Users/zacharykelly/Documents/MATLAB/localHookFolder/gkaModelEyeLocalHook.m".
  Hook success with status 0.
Checking for "gif" local hook.
Checking for "ExampleTestToolbox" local hook.
Checking for "combiLEDToolbox" local hook.
  Running local hook "/Users/zacharykelly/Documents/MATLAB/localHookFolder/combiLEDToolboxLocalHook.m".
  Hook success with status 0.
Checking for "BrainardLabToolbox" local hook.
Checking for "WatsonYellott2012_PupilSize" local hook.
Checking for "export_fig" local hook.
Checking for "Psychtoolbox-3" local hook.
Checking for "SilentSubstitutionToolbox" local hook.
Looks good: 14 resolved toolboxes deployed OK.


Processing Subjects:   0%|          | 0/6 [00:00<?, ?it/s]

Processing Activities:   0%|          | 0/7 [00:00<?, ?it/s]

Input: 
	 Subject id: FLIC_2006
	 Subject id number: 2006
	 Activity: walkOutdoor
	 Projection Type: justProjection
Output: 
	 Output dir: /Users/zacharykelly/Aguirre-Brainard Lab Dropbox/Zachary Kelly/FLIC_analysis/lightLogger/NEWscriptedIndoorOutdoorVideos2026/L-M/FLIC_2006/walkOutdoor
Subject: 1/1
Activity: 1/1
Processing chunk: 1/478...Starting parallel pool (parpool) using the 'Processes' profile ...
Connected to parallel pool with 28 workers.
read: 100.21s, process: 1.52s
Processing chunk: 2/478...read: 1.59s, process: 1.19s
Processing chunk: 3/478...read: 1.52s, process: 1.09s
Processing chunk: 4/478...read: 1.73s, process: 1.15s
Processing chunk: 5/478...read: 1.73s, process: 1.13s
Processing chunk: 6/478...read: 1.71s, process: 1.18s
Processing chunk: 7/478...read: 1.64s, process: 1.10s
Processing chunk: 8/478...read: 1.68s, process: 1.14s
Processing chunk: 9/478...read: 1.71s, process: 1.14s
Processing chunk: 10/478...read: 1.67s, process: 1.17s
Processing chunk: 11/478...rea

### 2. Plot the indivudal SPDs (optionally if not done above)
We also have this as a separate optional step because the first step is ridiculously long in terms of runtime

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.adjust_spd_axes(overwrite_existing=True, verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"], combine_figures=True)

#### 3. For each subject, output a grouped plot of their results for all activities


In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.group_spds_per_subject(overwrite_existing=True, verbose=True, subjects_to_skip=[2005], activities_to_skip=["gazeCalibration", "lunch", "phone"])

#### 4. Generate average SPDs for each activity by averaging across subjects for that activity

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_subject(overwrite_existing=True, verbose=True, subjects_to_skip=[2005], activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                                    combine_figures=True, common_axes=True)

#### 5. Group the mean SPDs for each activity by the activity type


In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.group_spds_across_subjects(overwrite_existing=True, verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"])

#### 6. Generate average SPDs across all activities' averages

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_all(overwrite_existing=True, verbose=True, activities_to_skip=["gazeCalibration", "lunch", "phone"], combine_figures=True, common_axes=True)

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_spds_across_groups(overwrite_existing=True, 
                                                   verbose=True, 
                                                   activities_to_skip=["gazeCalibration", "lunch", "phone"], 
                                                   combine_figures=True, 
                                                   common_axes=True,
                                                   groups={"sitting": ["sitBiopond", "work", "chat"], 
                                                           "walking": ["walkBiopond", "walkIndoor", "walkOutdoor"]}
                                                )

### Generate actigraphy graphs

In [ ]:
importlib.reload(preprocessing_pipeline)
preprocessing_pipeline.generate_actigraphy_graphs(overwrite_exsiting=True, 
                                                  verbose=True, 
                                                  subjects_to_skip=[2005]
                                                )

In [ ]:
import matlab.engine
eng = matlab.engine.start_matlab()

print(eng.version(nargout=1))